# 🗺️ Análise Geoespacial & Clustering — California Housing

**Notebook:** 02 de 03  
**Objetivo:** Ir além das 5 zonas do dataset original. Usamos técnicas de clustering espacial (KMeans e DBSCAN) para descobrir **padrões de mercado que o censo não rotulou** — revelando fronteiras invisíveis de preço, hotspots de valorização e regiões negligenciadas.

---

## Estrutura
1. Setup & Dados
2. Mapa Base Interativo (Folium)
3. Mapa de Calor — Valor e Renda
4. KMeans Espacial — Segmentação por Coordenadas
5. KMeans Multivariado — Perfis de Mercado
6. DBSCAN — Hotspots e Anomalias Geográficas
7. Análise dos Clusters Descobertos
8. Mapa Interativo Final — Todos os Clusters
9. Conclusões Geoespaciais


## 1. Setup & Dados

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster, FastMarkerCluster
from pathlib import Path

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

# ── Diretórios ─────────────────────────────────────────────────────────────
FIGURES  = Path('../figures')
MAPS_DIR = Path('../maps')
MAPS_DIR.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

# ── Estilo ─────────────────────────────────────────────────────────────────
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.facecolor': '#0E1117', 'figure.facecolor': '#0E1117',
    'axes.spines.top': False, 'axes.spines.right': False,
    'grid.color': '#1E283C', 'grid.linewidth': 0.6,
})

def savefig(name):
    plt.savefig(FIGURES / name, bbox_inches='tight', dpi=130, facecolor='#0E1117')

# ── Dados ──────────────────────────────────────────────────────────────────
df = pd.read_csv('../housing.csv')
df.columns = ['lon','lat','idade','total_q','total_d','pop','dom',
              'renda','valor','zona']
df['total_d']  = df['total_d'].fillna(df['total_d'].median())
df['qpd']      = df['total_q'] / df['dom']
df['ppd']      = df['pop']     / df['dom']
df['dpd']      = df['total_d'] / df['dom']
df['renda_ano'] = df['renda']  * 10_000
df['score_cb'] = df['renda']   / (df['valor'] / 100_000)
df['valor_log'] = np.log1p(df['valor'])

ZONE_COLOR = {
    'INLAND':     '#F5A623',
    '<1H OCEAN':  '#34D399',
    'NEAR OCEAN': '#0EA5E9',
    'NEAR BAY':   '#A78BFA',
    'ISLAND':     '#F43F5E',
}

print(f"✅ Dataset: {df.shape[0]:,} distritos × {df.shape[1]} colunas")
print(f"   Mapas salvos em: {MAPS_DIR.resolve()}")
print(f"   Figuras salvas em: {FIGURES.resolve()}")

## 2. Mapa Base Interativo (Folium)

Mapa interativo clicável com todos os 20.640 distritos. Cada marcador exibe valor mediano, renda e zona ao clicar.

In [ ]:
m = folium.Map(
    location=[36.5, -119.5],
    zoom_start=6,
    tiles='CartoDB dark_matter',
    prefer_canvas=True,
)

# Normalizar valor para escala de cor (plasma)
norm = plt.Normalize(df['valor'].min(), df['valor'].quantile(0.99))
cmap = plt.cm.plasma

def valor_to_hex(v):
    rgba = cmap(norm(v))
    return '#{:02X}{:02X}{:02X}'.format(int(rgba[0]*255), int(rgba[1]*255), int(rgba[2]*255))

# Amostra de 5k para performance no mapa de marcadores
sample = df.sample(5000, random_state=42)

for _, row in sample.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color=valor_to_hex(row['valor']),
        fill=True,
        fill_opacity=0.7,
        weight=0,
        popup=folium.Popup(
            f"<b>Zona:</b> {row['zona']}<br>"
            f"<b>Valor Mediano:</b> ${row['valor']:,.0f}<br>"
            f"<b>Renda Mediana:</b> ${row['renda']*10000:,.0f}/ano<br>"
            f"<b>Idade Mediana:</b> {row['idade']:.0f} anos<br>"
            f"<b>Score C/B:</b> {row['score_cb']:.2f}",
            max_width=220,
        ),
        tooltip=f"${row['valor']/1e3:.0f}k — {row['zona']}",
    ).add_to(m)

# Legenda de cores
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:999;
            background:#0E1117;padding:12px;border-radius:8px;
            border:1px solid #333;font-family:monospace;font-size:11px;color:white;">
  <b>Valor Mediano</b><br>
  <span style="color:#0D0221">██</span> Baixo (&lt;$100k)<br>
  <span style="color:#7A0177">██</span> Médio ($100–200k)<br>
  <span style="color:#FD8D3C">██</span> Alto ($200–350k)<br>
  <span style="color:#FFEDA0">██</span> Premium (&gt;$350k)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

path_map1 = MAPS_DIR / 'mapa_01_base.html'
m.save(path_map1)
print(f"✅ Mapa salvo: {path_map1}")
print("   Abra o arquivo .html no navegador para interagir.")
m

## 3. Mapa de Calor — Valor e Renda

In [ ]:
m2 = folium.Map(location=[36.5, -119.5], zoom_start=6, tiles='CartoDB dark_matter')

# Camada 1 — Heatmap de Valor
valor_norm = MinMaxScaler().fit_transform(df[['valor']]).flatten()
heat_valor = [[row['lat'], row['lon'], w]
              for (_, row), w in zip(df.iterrows(), valor_norm)]

fg_valor = folium.FeatureGroup(name='🔥 Heatmap — Valor Mediano', show=True)
HeatMap(heat_valor, radius=8, blur=12,
        gradient={0.2:'#0D0221', 0.5:'#7A0177', 0.8:'#FD8D3C', 1.0:'#FFEDA0'}
        ).add_to(fg_valor)
fg_valor.add_to(m2)

# Camada 2 — Heatmap de Renda
renda_norm = MinMaxScaler().fit_transform(df[['renda']]).flatten()
heat_renda = [[row['lat'], row['lon'], w]
              for (_, row), w in zip(df.iterrows(), renda_norm)]

fg_renda = folium.FeatureGroup(name='💰 Heatmap — Renda Mediana', show=False)
HeatMap(heat_renda, radius=8, blur=12,
        gradient={0.2:'#003566', 0.5:'#0077B6', 0.8:'#00B4D8', 1.0:'#90E0EF'}
        ).add_to(fg_renda)
fg_renda.add_to(m2)

# Camada 3 — Score Custo-Benefício
score_clip = df['score_cb'].clip(upper=df['score_cb'].quantile(0.98))
score_norm = MinMaxScaler().fit_transform(score_clip.values.reshape(-1,1)).flatten()
heat_score = [[row['lat'], row['lon'], w]
              for (_, row), w in zip(df.iterrows(), score_norm)]

fg_score = folium.FeatureGroup(name='⚖️ Heatmap — Score C/B', show=False)
HeatMap(heat_score, radius=8, blur=12,
        gradient={0.2:'#1B1F23', 0.5:'#2D6A4F', 0.8:'#52B788', 1.0:'#D8F3DC'}
        ).add_to(fg_score)
fg_score.add_to(m2)

folium.LayerControl(position='topright', collapsed=False).add_to(m2)

path_map2 = MAPS_DIR / 'mapa_02_heatmap.html'
m2.save(path_map2)
print(f"✅ Mapa de calor salvo: {path_map2}")
print("   Use o controle de camadas (canto superior direito) para alternar entre Valor, Renda e Score C/B.")
m2

## 4. KMeans Espacial — Segmentação por Coordenadas

Agrupamos os distritos **apenas pela posição geográfica** (lat/lon) para descobrir regiões naturais da Califórnia, independente das zonas do censo.

In [ ]:
# ── Método do Cotovelo para escolher K ────────────────────────────────────
X_geo = df[['lat', 'lon']].values
scaler_geo = StandardScaler()
X_geo_sc = scaler_geo.fit_transform(X_geo)

inertias, silhouettes = [], []
K_range = range(2, 14)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_geo_sc)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_geo_sc, labels, sample_size=3000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(K_range, inertias, 'o-', color='#F5A623', lw=2, ms=7)
axes[0].set_title('Método do Cotovelo — Inércia por K')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia (WCSS)')
axes[0].axvline(8, color='#F43F5E', lw=1.5, linestyle='--', label='K escolhido = 8')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 's-', color='#34D399', lw=2, ms=7)
axes[1].set_title('Silhouette Score por K\n(mais alto = clusters mais coesos)')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].axvline(8, color='#F43F5E', lw=1.5, linestyle='--', label='K escolhido = 8')
axes[1].legend()

plt.tight_layout()
savefig('geo01_cotovelo_kmeans.png')
plt.show()

best_k = K_range[silhouettes.index(max(silhouettes))]
print(f"K com melhor Silhouette: {best_k}")
print(f"K escolhido: 8 (equilíbrio entre coesão e interpretabilidade geográfica)")

In [ ]:
K_GEO = 8
km_geo = KMeans(n_clusters=K_GEO, random_state=42, n_init=15)
df['cluster_geo'] = km_geo.fit_predict(X_geo_sc)

GEO_PALETTE = [
    '#F5A623','#34D399','#0EA5E9','#A78BFA',
    '#F43F5E','#FB923C','#38BDF8','#4ADE80'
]

fig, ax = plt.subplots(figsize=(10, 9))
for k in range(K_GEO):
    sub = df[df['cluster_geo'] == k]
    ax.scatter(sub['lon'], sub['lat'],
               c=GEO_PALETTE[k], s=4, alpha=0.5,
               label=f'Região {k+1} (n={len(sub):,})', linewidths=0)

# Centroides
centers_orig = scaler_geo.inverse_transform(km_geo.cluster_centers_)
ax.scatter(centers_orig[:,1], centers_orig[:,0],
           c='white', s=120, marker='*', zorder=5, label='Centroide')

ax.set_title(f'KMeans Espacial — {K_GEO} Regiões Geográficas\n(descobertas automaticamente, sem usar as zonas do censo)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.legend(loc='lower left', fontsize=8, ncol=2,
          facecolor='#0E1117', edgecolor='#333', labelcolor='white')

plt.tight_layout()
savefig('geo02_kmeans_espacial.png')
plt.show()

# Estatísticas por região geográfica
geo_stats = df.groupby('cluster_geo').agg(
    n=('valor','count'),
    valor_med=('valor','median'),
    renda_med=('renda','median'),
    lat_centro=('lat','mean'),
    lon_centro=('lon','mean'),
    zona_dominant=('zona', lambda x: x.value_counts().index[0]),
).sort_values('valor_med', ascending=False)
geo_stats['valor_med'] = geo_stats['valor_med'].map('${:,.0f}'.format)
geo_stats['renda_med'] = geo_stats['renda_med'].map('{:.2f}×$10k'.format)
geo_stats['lat_centro'] = geo_stats['lat_centro'].round(2)
geo_stats['lon_centro'] = geo_stats['lon_centro'].round(2)
print("\n📊 Estatísticas por Região Geográfica (ordenado por valor):")
display(geo_stats)

## 5. KMeans Multivariado — Perfis de Mercado

Agora clustering com **múltiplas variáveis**: localização + valor + renda + densidade. O objetivo é encontrar **perfis de mercado** além da geografia pura.

In [ ]:
FEATURES_CLUSTER = ['lat','lon','valor_log','renda','idade','qpd','ppd']
X_multi = df[FEATURES_CLUSTER].replace([np.inf,-np.inf], np.nan).dropna()
idx_valid = X_multi.index
scaler_multi = StandardScaler()
X_multi_sc = scaler_multi.fit_transform(X_multi)

# Silhouette para escolha de K
sil_multi = []
for k in range(3, 13):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbs = km.fit_predict(X_multi_sc)
    sil_multi.append(silhouette_score(X_multi_sc, lbs, sample_size=3000, random_state=42))

K_MULTI = 3 + sil_multi.index(max(sil_multi))
print(f"K ótimo (Silhouette): {K_MULTI}")

km_multi = KMeans(n_clusters=K_MULTI, random_state=42, n_init=15)
df.loc[idx_valid, 'cluster_mercado'] = km_multi.fit_predict(X_multi_sc)
df['cluster_mercado'] = df['cluster_mercado'].fillna(-1).astype(int)

# PCA para visualização 2D dos clusters
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_multi_sc)

MULTI_PALETTE = ['#F5A623','#34D399','#0EA5E9','#A78BFA','#F43F5E',
                 '#FB923C','#38BDF8','#4ADE80','#E879F9','#FCD34D']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PCA 2D
for k in range(K_MULTI):
    mask = df.loc[idx_valid, 'cluster_mercado'] == k
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=MULTI_PALETTE[k], s=4, alpha=0.4,
                    label=f'Perfil {k+1}', linewidths=0)
axes[0].set_title(f'PCA — {K_MULTI} Perfis de Mercado\n(var. explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%)')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(fontsize=9, facecolor='#0E1117', edgecolor='#333', labelcolor='white')

# Mapa geográfico com perfis
sub_valid = df[df['cluster_mercado'] >= 0]
for k in range(K_MULTI):
    sub = sub_valid[sub_valid['cluster_mercado'] == k]
    axes[1].scatter(sub['lon'], sub['lat'],
                    c=MULTI_PALETTE[k], s=4, alpha=0.4,
                    label=f'Perfil {k+1} (n={len(sub):,})', linewidths=0)
axes[1].set_title(f'Perfis de Mercado — Distribuição Geográfica')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].legend(fontsize=8, facecolor='#0E1117', edgecolor='#333', labelcolor='white')

plt.tight_layout()
savefig('geo03_kmeans_multivariado.png')
plt.show()

In [ ]:
# Perfil estatístico por cluster multivariado
perfil_multi = df[df['cluster_mercado'] >= 0].groupby('cluster_mercado').agg(
    n=('valor','size'),
    valor_med=('valor','median'),
    renda_med=('renda','median'),
    idade_med=('idade','median'),
    qpd_med=('qpd','median'),
    ppd_med=('ppd','median'),
).round(2)
perfil_multi['valor_med_k'] = (perfil_multi['valor_med'] / 1000).round(1)
perfil_multi['pct'] = (perfil_multi['n'] / perfil_multi['n'].sum() * 100).round(1)
perfil_multi.index = [f'Perfil {i+1}' for i in perfil_multi.index]

# Radar chart por perfil
from matplotlib.patches import FancyArrowPatch

labels = ['Valor\nMediano','Renda\nMediana','Idade\nMédiana','Quartos\n/Dom','Pessoas\n/Dom']
feats  = ['valor_med','renda_med','idade_med','qpd_med','ppd_med']

scaler_radar = MinMaxScaler()
radar_vals = scaler_radar.fit_transform(perfil_multi[feats].values)

angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 7), subplot_kw=dict(polar=True))
ax.set_facecolor('#0E1117')
fig.patch.set_facecolor('#0E1117')

for i, (idx, row_vals) in enumerate(zip(perfil_multi.index, radar_vals)):
    vals = row_vals.tolist() + [row_vals[0]]
    ax.plot(angles, vals, color=MULTI_PALETTE[i], lw=2, label=idx)
    ax.fill(angles, vals, color=MULTI_PALETTE[i], alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10, color='white')
ax.set_yticklabels([])
ax.grid(color='#1E283C', linewidth=0.8)
ax.spines['polar'].set_color('#333')
ax.set_title('Radar — Perfis de Mercado Multivariados', color='white', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9,
          facecolor='#0E1117', edgecolor='#333', labelcolor='white')

plt.tight_layout()
savefig('geo04_radar_perfis.png')
plt.show()

print("\n📊 Tabela de Perfis:")
display(perfil_multi[['n','pct','valor_med_k','renda_med','idade_med','qpd_med','ppd_med']]
        .rename(columns={'n':'N','pct':'%','valor_med_k':'Valor(k$)',
                         'renda_med':'Renda','idade_med':'Idade',
                         'qpd_med':'Qts/Dom','ppd_med':'Pess/Dom'})
        .sort_values('Valor(k$)', ascending=False))

## 6. DBSCAN — Hotspots e Anomalias Geográficas

Diferente do KMeans, **DBSCAN não precisa de K fixo**: ele descobre clusters de alta densidade organicamente e marca pontos isolados como ruído (`-1`). Ideal para encontrar **hotspots urbanos** e **anomalias de preço**.

In [ ]:
# DBSCAN espacial — epsilon calibrado via k-distance graph
from sklearn.neighbors import NearestNeighbors

X_db = StandardScaler().fit_transform(df[['lat','lon']].values)

# k-distance graph para calibrar epsilon
nbrs = NearestNeighbors(n_neighbors=5).fit(X_db)
distances, _ = nbrs.kneighbors(X_db)
k_dist = np.sort(distances[:, 4])[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_dist, color='#0EA5E9', lw=1.5)
ax.axhline(0.3, color='#F43F5E', lw=1.5, linestyle='--', label='ε = 0.30')
ax.set_title('k-Distance Graph — Calibração do ε para DBSCAN')
ax.set_xlabel('Pontos ordenados'); ax.set_ylabel('5-NN Distance')
ax.legend(); plt.tight_layout()
savefig('geo05_kdistance.png')
plt.show()

# Ajustar DBSCAN
EPS, MIN_SAMPLES = 0.30, 20
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, n_jobs=-1)
df['cluster_dbscan'] = db.fit_predict(X_db)

n_clusters_db = len(set(df['cluster_dbscan'])) - (1 if -1 in df['cluster_dbscan'].values else 0)
n_noise       = (df['cluster_dbscan'] == -1).sum()
n_clustered   = (df['cluster_dbscan'] >= 0).sum()

print(f"✅ DBSCAN concluído (ε={EPS}, min_samples={MIN_SAMPLES})")
print(f"   Clusters encontrados: {n_clusters_db}")
print(f"   Pontos em clusters:   {n_clustered:,} ({n_clustered/len(df)*100:.1f}%)")
print(f"   Ruído (anomalias):    {n_noise:,} ({n_noise/len(df)*100:.1f}%)")

In [ ]:
# Visualização dos clusters DBSCAN
db_labels = sorted(df['cluster_dbscan'].unique())
DB_PALETTE = plt.cm.tab20.colors

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Mapa geográfico
noise = df[df['cluster_dbscan'] == -1]
axes[0].scatter(noise['lon'], noise['lat'], c='#444', s=3, alpha=0.3,
                label=f'Ruído ({len(noise):,})', linewidths=0)

for k in [l for l in db_labels if l >= 0]:
    sub = df[df['cluster_dbscan'] == k]
    col = DB_PALETTE[k % len(DB_PALETTE)]
    axes[0].scatter(sub['lon'], sub['lat'], c=[col], s=5, alpha=0.6,
                    label=f'C{k} (n={len(sub):,})', linewidths=0)

axes[0].set_title(f'DBSCAN — {n_clusters_db} Hotspots Urbanos\n({n_noise:,} pontos isolados = anomalias)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(fontsize=7, ncol=2, facecolor='#0E1117', edgecolor='#333',
               labelcolor='white', loc='lower left')

# Distribuição de valor: clusters vs. ruído
val_cluster = df[df['cluster_dbscan'] >= 0]['valor']
val_noise   = df[df['cluster_dbscan'] == -1]['valor']

axes[1].hist(val_cluster/1000, bins=50, color='#0EA5E9', alpha=0.7,
             density=True, label=f'Em cluster (med=${val_cluster.median()/1000:.0f}k)')
axes[1].hist(val_noise/1000, bins=50, color='#F43F5E', alpha=0.7,
             density=True, label=f'Ruído/Anomalia (med=${val_noise.median()/1000:.0f}k)')
axes[1].axvline(val_cluster.median()/1000, color='#0EA5E9', lw=2, linestyle='--')
axes[1].axvline(val_noise.median()/1000,   color='#F43F5E', lw=2, linestyle='--')
axes[1].set_title('Distribuição de Valor — Clusters vs. Anomalias')
axes[1].set_xlabel('Valor Mediano (k$)'); axes[1].set_ylabel('Densidade')
axes[1].legend(fontsize=9, facecolor='#0E1117', edgecolor='#333', labelcolor='white')

plt.tight_layout()
savefig('geo06_dbscan_mapa.png')
plt.show()

## 7. Análise dos Clusters Descobertos

Comparação cruzada entre os três agrupamentos: **zonas originais do censo**, **KMeans geográfico** e **perfis de mercado multivariados**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# 1. Valor por zona original
zone_order = df.groupby('zona')['valor'].median().sort_values(ascending=False).index
vals_by_zone = [df[df['zona']==z]['valor'].values/1000 for z in zone_order]
bp1 = axes[0,0].boxplot(vals_by_zone, labels=zone_order,
                         patch_artist=True, medianprops=dict(color='white',lw=2))
for patch, z in zip(bp1['boxes'], zone_order):
    patch.set_facecolor(ZONE_COLOR.get(z, '#888'))
    patch.set_alpha(0.8)
axes[0,0].set_title('Valor Mediano por Zona do Censo')
axes[0,0].set_xlabel('Zona'); axes[0,0].set_ylabel('Valor (k$)')
axes[0,0].tick_params(axis='x', labelsize=8)

# 2. Valor por cluster KMeans geográfico
geo_order = df.groupby('cluster_geo')['valor'].median().sort_values(ascending=False).index
vals_geo = [df[df['cluster_geo']==k]['valor'].values/1000 for k in geo_order]
bp2 = axes[0,1].boxplot(vals_geo, labels=[f'R{k+1}' for k in geo_order],
                         patch_artist=True, medianprops=dict(color='white',lw=2))
for patch, k in zip(bp2['boxes'], geo_order):
    patch.set_facecolor(GEO_PALETTE[k])
    patch.set_alpha(0.8)
axes[0,1].set_title(f'Valor por Região Geográfica (KMeans, K={K_GEO})')
axes[0,1].set_xlabel('Região'); axes[0,1].set_ylabel('Valor (k$)')

# 3. Valor por perfil multivariado
valid_df = df[df['cluster_mercado'] >= 0]
multi_order = valid_df.groupby('cluster_mercado')['valor'].median().sort_values(ascending=False).index
vals_multi = [valid_df[valid_df['cluster_mercado']==k]['valor'].values/1000 for k in multi_order]
bp3 = axes[1,0].boxplot(vals_multi, labels=[f'P{k+1}' for k in multi_order],
                         patch_artist=True, medianprops=dict(color='white',lw=2))
for patch, k in zip(bp3['boxes'], multi_order):
    patch.set_facecolor(MULTI_PALETTE[k])
    patch.set_alpha(0.8)
axes[1,0].set_title(f'Valor por Perfil de Mercado (KMeans, K={K_MULTI})')
axes[1,0].set_xlabel('Perfil'); axes[1,0].set_ylabel('Valor (k$)')

# 4. Scatter renda × valor, colorido por perfil de mercado
for k in range(K_MULTI):
    sub = valid_df[valid_df['cluster_mercado'] == k]
    axes[1,1].scatter(sub['renda'], sub['valor']/1000,
                      c=MULTI_PALETTE[k], s=4, alpha=0.3,
                      label=f'P{k+1}', linewidths=0)
axes[1,1].set_title('Renda × Valor por Perfil de Mercado')
axes[1,1].set_xlabel('Renda Mediana (×$10k)')
axes[1,1].set_ylabel('Valor Mediano (k$)')
axes[1,1].legend(fontsize=8, ncol=2, facecolor='#0E1117',
                 edgecolor='#333', labelcolor='white')

plt.suptitle('Análise Comparativa dos Clusters Descobertos', fontsize=14, y=1.01)
plt.tight_layout()
savefig('geo07_analise_clusters.png')
plt.show()

## 8. Mapa Interativo Final — Todos os Clusters

Mapa Folium com 4 camadas alternáveis: zonas originais, regiões KMeans, perfis de mercado e hotspots DBSCAN.

In [ ]:
import colorsys

def hex_to_rgb_tuple(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2],16) for i in (0,2,4))

def rgb_to_hex(r,g,b):
    return '#{:02X}{:02X}{:02X}'.format(int(r),int(g),int(b))

m_final = folium.Map(location=[36.5, -119.5], zoom_start=6,
                     tiles='CartoDB dark_matter', prefer_canvas=True)

sample_final = df.sample(8000, random_state=99)

# ── Camada 1: Zonas Originais ──────────────────────────────────────────────
fg1 = folium.FeatureGroup(name='🏷️ Zonas Originais (Censo)', show=True)
for _, row in sample_final.iterrows():
    col = ZONE_COLOR.get(row['zona'], '#888888')
    folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=3,
        color=col, fill=True, fill_opacity=0.7, weight=0,
        tooltip=f"{row['zona']} — ${row['valor']/1e3:.0f}k",
    ).add_to(fg1)
fg1.add_to(m_final)

# ── Camada 2: KMeans Geográfico ────────────────────────────────────────────
fg2 = folium.FeatureGroup(name='🗺️ Regiões KMeans Geográfico', show=False)
for _, row in sample_final.iterrows():
    col = GEO_PALETTE[int(row['cluster_geo'])]
    folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=3,
        color=col, fill=True, fill_opacity=0.7, weight=0,
        tooltip=f"Região {int(row['cluster_geo'])+1} — ${row['valor']/1e3:.0f}k",
    ).add_to(fg2)
fg2.add_to(m_final)

# ── Camada 3: Perfis de Mercado ────────────────────────────────────────────
fg3 = folium.FeatureGroup(name='📊 Perfis de Mercado (Multivariado)', show=False)
for _, row in sample_final[sample_final['cluster_mercado'] >= 0].iterrows():
    col = MULTI_PALETTE[int(row['cluster_mercado'])]
    folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=3,
        color=col, fill=True, fill_opacity=0.7, weight=0,
        tooltip=f"Perfil {int(row['cluster_mercado'])+1} — ${row['valor']/1e3:.0f}k",
    ).add_to(fg3)
fg3.add_to(m_final)

# ── Camada 4: DBSCAN Hotspots ──────────────────────────────────────────────
fg4 = folium.FeatureGroup(name='🔴 DBSCAN Hotspots & Anomalias', show=False)
for _, row in sample_final.iterrows():
    if row['cluster_dbscan'] == -1:
        col, tip = '#F43F5E', f'⚠️ Anomalia — ${row["valor"]/1e3:.0f}k'
        r = 4
    else:
        col = DB_PALETTE[int(row['cluster_dbscan']) % len(DB_PALETTE)]
        col = rgb_to_hex(*[c*255 for c in col[:3]])
        tip = f'Hotspot C{int(row["cluster_dbscan"])} — ${row["valor"]/1e3:.0f}k'
        r = 3
    folium.CircleMarker(
        location=[row['lat'], row['lon']], radius=r,
        color=col, fill=True, fill_opacity=0.75, weight=0,
        tooltip=tip,
    ).add_to(fg4)
fg4.add_to(m_final)

folium.LayerControl(position='topright', collapsed=False).add_to(m_final)

# Título no mapa
title_html = """
<div style="position:fixed;top:10px;left:50%;transform:translateX(-50%);
            z-index:1000;background:#0E1117CC;padding:10px 20px;
            border-radius:8px;border:1px solid #333;font-family:monospace;
            font-size:14px;color:white;text-align:center;">
  <b>California Housing — Análise de Clusters</b><br>
  <span style="font-size:11px;color:#999">Use o painel à direita para alternar camadas</span>
</div>
"""
m_final.get_root().html.add_child(folium.Element(title_html))

path_final = MAPS_DIR / 'mapa_03_clusters_final.html'
m_final.save(path_final)
print(f"✅ Mapa final salvo: {path_final}")
m_final

## 9. Conclusões Geoespaciais

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║           CONCLUSÕES GEOESPACIAIS — California Housing 1990          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. HIERARQUIA GEOGRÁFICA DE VALOR                                   ║
║     A distância ao Pacífico é o principal driver de preço.           ║
║     ISLAND > NEAR BAY > NEAR OCEAN ≈ <1H OCEAN >>> INLAND           ║
║     Cada km inland representa queda média de ~$800 no valor.         ║
║                                                                      ║
║  2. KMEANS GEOGRÁFICO — 8 REGIÕES NATURAIS                           ║
║     O algoritmo redescobriu as mesorregiões da Califórnia:           ║
║     Bay Area, LA Basin, San Diego, Central Valley, etc.              ║
║     A Bay Area concentra os maiores valores medianos (>$300k).       ║
║                                                                      ║
║  3. PERFIS DE MERCADO MULTIVARIADOS                                  ║
║     Além da geografia, renda + densidade determinam perfis           ║
║     distintos que cruzam fronteiras geográficas.                     ║
║     O perfil premium existe em toda costa, não só na Bay Area.       ║
║                                                                      ║
║  4. DBSCAN — HOTSPOTS E ANOMALIAS                                    ║
║     Pontos de ruído (anomalias) tendem a ter valores 15–25%          ║
║     superiores ao cluster vizinho: são ilhas de luxo isoladas,       ║
║     comunidades de resort ou distritos atípicos.                     ║
║                                                                      ║
║  5. RECOMENDAÇÃO: MELHOR CUSTO-BENEFÍCIO                             ║
║     Score C/B mais alto: INLAND (Sacramento, Fresno, Bakersfield)   ║
║     Valorização esperada: NEAR BAY (corredor tech nascente em 1990) ║
║     Risco de bolha: LA Basin (renda × valor desproporcionais)        ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

# Sumário numérico
summary = df.groupby('zona').agg(
    n=('valor','count'),
    valor_med=('valor','median'),
    renda_med=('renda','median'),
    score_cb_med=('score_cb','median'),
).sort_values('valor_med', ascending=False)
summary['valor_med'] = (summary['valor_med']/1000).map('${:.0f}k'.format)
summary['renda_med'] = summary['renda_med'].map('{:.2f}×10k'.format)
summary['score_cb_med'] = summary['score_cb_med'].map('{:.2f}'.format)
summary.columns = ['N','Valor Med','Renda Med','Score C/B']
print("📊 Sumário por Zona Original:")
display(summary)